# Practice 4 — Solution: a systematic comparison

*Exposome Analytics Summer School, London 2026 — Deep Learning day.*

You were asked to build your own network on MNIST and try to beat the first
result. This notebook does it methodically: the **same training budget** for
every configuration, so the comparison is fair.

> **Switch to a GPU first:** Runtime → Change runtime type → T4 GPU.


## The data, exactly as in the practical


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from keras.datasets import mnist
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Input
from keras.callbacks import EarlyStopping
from keras.utils import to_categorical

(train_X, train_Y), (test_X, test_Y) = mnist.load_data()

train_X = train_X.astype('float32').reshape(train_X.shape[0], -1) / 255
test_X  = test_X.astype('float32').reshape(test_X.shape[0], -1) / 255

train_label = to_categorical(train_Y)
test_label  = to_categorical(test_Y)

# train / validation split, 80 / 20
n_val = int(0.2 * train_X.shape[0])
valid_X, valid_label = train_X[:n_val], train_label[:n_val]
tr_X,    tr_label    = train_X[n_val:], train_label[n_val:]

print('train', tr_X.shape, ' validation', valid_X.shape, ' test', test_X.shape)


## One function to build, train and score a configuration

Every configuration gets the same budget: at most 30 epochs, early stopping
with a patience of 3, and the weights of the best epoch are restored. This is
what makes the comparison honest: a model is never penalised for converging
sooner.


In [ ]:
INPUT_DIM = tr_X.shape[1]

def build(units=(128, 64), dropout=0.0, batchnorm=False):
    model = Sequential()
    model.add(Input(shape=(INPUT_DIM,)))
    for u in units:
        model.add(Dense(u, activation='relu'))
        if batchnorm:
            model.add(BatchNormalization())
        if dropout > 0:
            model.add(Dropout(dropout))
    model.add(Dense(10, activation='softmax'))
    model.compile(loss='categorical_crossentropy', optimizer='adam',
                  metrics=['accuracy'])
    return model

def run(name, units=(128, 64), dropout=0.0, batchnorm=False, batch_size=128):
    model = build(units, dropout, batchnorm)
    stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
    hist = model.fit(tr_X, tr_label, batch_size=batch_size, epochs=30, verbose=0,
                     validation_data=(valid_X, valid_label), callbacks=[stop])
    test_acc = model.evaluate(test_X, test_label, verbose=0)[1]
    print(f'{name:34s} test acc {test_acc:.4f}   '
          f'({len(hist.history["loss"]):2d} epochs, '
          f'{model.count_params():,} params)')
    return {'configuration': name,
            'test accuracy': round(float(test_acc), 4),
            'epochs used': len(hist.history['loss']),
            'parameters': model.count_params()}


## The configurations

Six models, from the baseline of the practical to combinations of the three
options. Each one takes a minute or two on a GPU.


In [ ]:
results = []
results.append(run('baseline  128-64',              units=(128, 64)))
results.append(run('+ dropout 0.2',                 units=(128, 64), dropout=0.2))
results.append(run('+ batch normalisation',         units=(128, 64), batchnorm=True))
results.append(run('+ dropout and batch norm',      units=(128, 64), dropout=0.2, batchnorm=True))
results.append(run('wider     512-256',             units=(512, 256), dropout=0.2))
results.append(run('deeper    256-128-64',          units=(256, 128, 64), dropout=0.2))


## The table


In [ ]:
table = pd.DataFrame(results).sort_values('test accuracy', ascending=False)
table = table.reset_index(drop=True)
table


## The same thing as a picture


In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
order = table.sort_values('test accuracy')
ax.barh(order['configuration'], order['test accuracy'], color='steelblue')
ax.set_xlim(0.96, 1.0)
ax.set_xlabel('test accuracy')
ax.set_title('Same budget for every configuration')
for i, (v, n) in enumerate(zip(order['test accuracy'], order['parameters'])):
    ax.text(v + 0.0008, i, f'{v:.4f}   ({n:,} par.)', va='center', fontsize=9)
plt.tight_layout(); plt.show()


---

## What to take away

**The differences are small, and that is the lesson.** All these networks land
between roughly 97% and 98.5%. Once the model is big enough for the task,
changing its size moves the result far less than one would expect.

**Dropout and batch normalisation earn their keep on the gap, not on the peak.**
Look at the number of epochs used: the regularised models keep improving for
longer before early stopping intervenes. They overfit later, which is exactly
what they are for.

**More parameters is not more accuracy.** The widest model here has several
times the parameters of the baseline for a fraction of a point, sometimes for
nothing at all. Compare the two columns of the table.

**And the real answer to the exercise is elsewhere.** To go clearly beyond this,
you need a model that knows the input is an *image*: a convolutional network.
That is the next notebook, and it beats everything here with fewer parameters.
